In [4]:
import os
import json
import pandas as pd
import numpy as np

from openai import OpenAI
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import Markdown, display

In [5]:
load_dotenv()

api_key = os.getenv('API_KEY')
if not api_key:
    raise ValueError("api key not found")

client = OpenAI(api_key=api_key)

In [6]:
available_models = client.models.list()
for model in available_models.data:
    print(model)

Model(id='gpt-5.4-mini-2026-03-17', created=1773451076, object='model', owned_by='system', shutdown_date=None)
Model(id='gpt-image-2', created=1776399795, object='model', owned_by='system', shutdown_date='2027-02-10')
Model(id='gpt-5.4-pro', created=1772659601, object='model', owned_by='system', shutdown_date=None)
Model(id='gpt-4o-mini-tts', created=1742403959, object='model', owned_by='system', shutdown_date='2027-01-06')
Model(id='gpt-realtime-mini-2025-12-15', created=1765612007, object='model', owned_by='system', shutdown_date='2027-01-20')
Model(id='text-embedding-ada-002', created=1671217299, object='model', owned_by='openai-internal', shutdown_date=None)
Model(id='gpt-4o-2024-08-06', created=1722814719, object='model', owned_by='system', shutdown_date='2027-01-14')
Model(id='gpt-realtime-2.1', created=1782254687, object='model', owned_by='system', shutdown_date=None)
Model(id='o3', created=1744225308, object='model', owned_by='system', shutdown_date='2026-12-11')
Model(id='gpt-

In [7]:
model='gpt-6-luna'

APP_INSTRUCTIONS = """
You are StudyMate, an educational assistant.

Your responsibilities:
1. Help students understand their lecture notes.
2. Explain concepts clearly and accurately.
3. Use examples when appropriate.
4. Avoid unnecessary technical jargon.
5. When answering questions based on notes, prioritize
   the provided material.
6. If the notes do not contain enough information,
   explicitly say so instead of inventing facts.
"""

def generate_response(prompt):
   response = client.responses.create(
      model=model,
      instructions=APP_INSTRUCTIONS,
      input=prompt
   )

   return response.output_text

In [8]:
prompt = """
Explain the difference between linear regression
and logistic regression.

Use simple examples.
"""

answer = generate_response(prompt)
display(Markdown(answer))

**Linear regression** predicts a **number**.  
- Example: Use a home’s size to estimate its price, such as **$320,000**.  
- Its predictions can range across a continuous scale.

**Logistic regression** predicts the **chance of a category**—often one of two outcomes.  
- Example: Estimate whether an email is spam. It might predict a **90% chance** of spam, which could then be classified as “spam” using a cutoff such as 50%.

**In short:** linear regression answers “How much?”; logistic regression answers “Which category?” or “How likely?”

In [9]:
def summarize_notes(notes):
    prompt = f"""
    Summarize the following lecture notes.

    Organize your response into:
    1. Main idea
    2. Important concepts
    3. Important formulas
    4. Relationships between concepts
    5. Common misconceptions

    Only use information from the notes.
    Do not invent additional facts.

    LECTURE NOTES:
    {notes}
    """

    return generate_response(prompt)

In [10]:
notes = """
Logistic Regression

Logistic regression is a supervised machine learning
algorithm used for classification.

It calculates a linear combination of input features:

z = w1*x1 + w2*x2 + ... + b

The sigmoid function transforms this value into
a probability between 0 and 1.

sigmoid(z) = 1 / (1 + exp(-z))

The probability is compared against a classification
threshold, commonly 0.5.

Maximum Likelihood Estimation is used to estimate
the model parameters.

The coefficients represent changes in log-odds.
Exponentiating a coefficient gives its odds ratio.

Logistic regression is called regression because
it estimates a continuous linear combination of
the input features, even though its final task
is classification.
"""

summary = summarize_notes(notes)
display(Markdown(summary))

## 1. Main idea
Logistic regression is a supervised machine learning algorithm used for classification. It combines input features linearly, then transforms the result into a probability.

## 2. Important concepts
- **Linear combination:** The model combines features using weights and a bias.
- **Sigmoid function:** Converts the linear combination into a value between 0 and 1.
- **Classification threshold:** The resulting probability is compared with a threshold, commonly 0.5.
- **Maximum Likelihood Estimation (MLE):** Used to estimate the model parameters.
- **Coefficients:** Represent changes in log-odds. Exponentiating a coefficient gives its odds ratio.

## 3. Important formulas
- Linear combination:  
  \[
  z = w_1x_1 + w_2x_2 + \cdots + b
  \]
- Sigmoid function:  
  \[
  \operatorname{sigmoid}(z) = \frac{1}{1+\exp(-z)}
  \]

## 4. Relationships between concepts
The model first calculates \(z\) from the input features, then applies the sigmoid function to produce a probability. That probability is compared with a classification threshold to make a classification. MLE is used to estimate the parameters in the model.

## 5. Common misconceptions
- Despite its name, logistic regression’s final task is classification.
- It is called “regression” because it estimates a continuous linear combination of input features.
- The sigmoid output is a probability, not the classification itself; classification uses a threshold.

In [11]:
def generate_flashcards(notes, number=8):
    prompt = f"""
    Create {number} useful flashcards based on these notes.

    Requirements:
    - Focus on important concepts.
    - Questions should test understanding, not just memorization.
    - Answers should be concise.
    - Do not introduce information absent from the notes.

    Return a JSON object with this structure:

    {{
        "flashcards": [
            {{
                "question": "Question here",
                "answer": "Answer here"
            }}
        ]
    }}

    NOTES:
    {notes}
    """

    response = client.responses.create(
        model=model,
        instructions=APP_INSTRUCTIONS,
        input=prompt,
        text={"format":{"type":"json_object"}}
    )

    return json.loads(response.output_text)

In [12]:
flashcards = generate_flashcards(notes, 8)
flashcards_df = pd.DataFrame(flashcards['flashcards'])
display(flashcards_df)

,question,answer
0,How does logistic regression turn input featur...,It computes a linear combination of the featur...
1,What role does the classification threshold pl...,It determines how the predicted probability is...
2,Why is the sigmoid function useful for classif...,It transforms the model's linear output into a...
3,How are the parameters of a logistic regressio...,Using Maximum Likelihood Estimation.
4,What do the coefficients in logistic regressio...,They represent changes in log-odds.
5,How can a logistic regression coefficient be e...,Exponentiate the coefficient.
6,Why is logistic regression called regression e...,It estimates a continuous linear combination o...
7,What makes logistic regression a supervised le...,It is a supervised machine learning algorithm ...


In [13]:
def generate_quiz(notes, number=4):
    prompt = f"""
    Create {number} multiple-choice questions based on
    the provided lecture notes.

    Requirements:
    - Four options per question.
    - Exactly one correct answer.
    - Test conceptual understanding.
    - Include an explanation of the correct answer.
    - answer_index must be an integer from 0 to 3.

    Return JSON in this format:

    {{
        "questions": [
            {{
                "question": "Question?",
                "options": [
                    "Option A",
                    "Option B",
                    "Option C",
                    "Option D"
                ],
                "answer_index": 0,
                "explanation": "Explanation..."
            }}
        ]
    }}

    NOTES:
    {notes}
    """

    response = client.responses.create(
        model=model,
        instructions=APP_INSTRUCTIONS,
        input=prompt,
        text={"format": {"type": "json_object"}}
    )

    return json.loads(response.output_text)

In [14]:
quiz = generate_quiz(notes, 4)
quiz_df = pd.DataFrame(quiz['questions'])
display(quiz_df)

,question,options,answer_index,explanation
0,What role does the sigmoid function play in lo...,[It converts the linear combination of feature...,0,The sigmoid function transforms the model's li...
1,How is the probability produced by logistic re...,[It is compared with a classification threshol...,0,Logistic regression compares its predicted pro...
2,What does exponentiating a logistic regression...,[The odds ratio associated with that coefficie...,0,A coefficient represents a change in log-odds;...
3,Why is logistic regression called “regression”...,[It estimates a continuous linear combination ...,0,The model first calculates a continuous linear...


In [15]:
def run_quiz(quiz):
    score = 0
    questions = quiz['questions']

    for i, question in enumerate(questions, start=1):
        print(f"\nQuestion {i}: {question['question']}")
        for j, option in enumerate(question['options']):
            print(f"{j + 1}. option")

        while True:
            try:
                answer = int(input('answer in numerical form(1-4): '))
                if answer in range(1, 5):
                    break
                print('choose a number between 1 and 4')

            except ValueError:
                print('enter number')
        
        if answer - 1 == question['answer_index']:
            print('correct')
            score += 1
        else:
            correct = question['options'][question['answer_index']]
            print(f'wrong, correct asnwer: {correct}')

        print(f'explanation: {question['explanation']}')

    print(f'final score: {score} / {len(questions)}')
    return score

In [16]:
flashcards_df.to_json('flashcards.json', orient='records', indent=4)
quiz_df.to_json('quiz.json', orient='records', indent=4)